# Paso 8 — Extracción del Herfindahl-Hirschman Market Concentration Index (WITS)

**TFM:** Multicausalidad de la inflación: un estudio comparativo con técnicas de Machine Learning

**Objetivo de este notebook:**
Extraer, vía API, el indicador *HH Market Concentration Index (Export)* para incorporarlo a la
dimensión estructural del proyecto 

**Nota:**
Este indicador Pertenece a **WITS (World Integrated Trade
Solution)**, una plataforma con API propia, operada conjuntamente por el Banco Mundial, UNCTAD y el ITC.



## Bloque 1: Importación de librerías

**Objetivo:** cargar las herramientas necesarias para hacer la petición HTTP a la API de WITS
y para organizar la respuesta en una tabla.

**Justificación metodológica:** a diferencia del notebook 01 (WDI vía `wbgapi`), aquí no existe
una librería de Python dedicada y mantenida para WITS, así que trabajamos directamente con `requests`
sobre la API REST pública, siguiendo el mismo principio de reproducibilidad (cualquiera que corra
este notebook obtiene la misma respuesta de la fuente oficial).


In [1]:
import requests
# requests permite hacer la llamada HTTP GET directamente a la API REST de WITS.

import pandas as pd

import json
# json nos sirve para inspeccionar "a mano" la estructura de la respuesta antes de parsearla,
# ya que el formato SDMX-JSON de WITS no es tan directo como el de wbgapi.

## Bloque 2: Definición de la consulta a la API

**Objetivo:** construir la URL de consulta para pedir el indicador `HH-MKT-CNCNTRTN-NDX`,
para todos los países reportantes disponibles (`reporter=all`) y todos los años disponibles
(`year=all`), en formato JSON.

**Justificación metodológica:** la API de WITS permite un máximo de dos dimensiones en "ALL"
simultáneamente. Aquí usamos exactamente dos (`reporter` y `year`), dejando el indicador fijo
en uno solo — por lo tanto la consulta es válida en una única llamada, sin necesidad de loops
por país ni por año (mucho más simple y reproducible que el enfoque país-por-país).

**Resultado esperado:** una variable `url_market_concentration` con la URL completa de la
consulta, lista para ejecutarse en el bloque siguiente.

In [2]:
base_url = "https://wits.worldbank.org/API/V1/SDMX/V21/datasource"
datasource = "tradestats-trade"
indicador = "HH-MKT-CNCNTRTN-NDX"

# reporter=all -> todos los países/agrupaciones reportantes disponibles en WITS
# year=all -> todo el rango temporal disponible (confirmado 1988-2025 en el data catalog)
# format=JSON -> pedimos la respuesta en JSON en vez de XML, para facilitar el parseo con pandas
url_market_concentration = (
    f"{base_url}/{datasource}/reporter/all/year/all/indicator/{indicador}?format=JSON"
)

print(url_market_concentration)

https://wits.worldbank.org/API/V1/SDMX/V21/datasource/tradestats-trade/reporter/all/year/all/indicator/HH-MKT-CNCNTRTN-NDX?format=JSON


## Bloque 3: Llamada a la API y verificación de la respuesta

**Objetivo:** ejecutar la petición HTTP y confirmar que la API respondió correctamente antes de
intentar procesar los datos.

**Justificación metodológica:** la documentación de WITS advierte que consultas demasiado grandes
pueden devolver error 400/413 ("The request yield large data to return"). Verificar el `status_code`
y el tamaño de la respuesta antes de parsear evita que un error de red se confunda más adelante con
un problema de datos faltantes.

**Resultado esperado:** `status_code` igual a 200, y un objeto `respuesta_json` con el contenido
de la respuesta.

Es importante tener ACCESO A INTERNET, sino puede dar error

In [4]:
response = requests.get(url_market_concentration, timeout=60)

print("Status code:", response.status_code)
print("Tamaño de la respuesta (bytes):", len(response.content))

if response.status_code == 200:
    respuesta_json = response.json()
else:
    # Si falla, mostramos el texto de error que devuelve WITS para poder diagnosticar
    print(response.text[:2000])

Status code: 200
Tamaño de la respuesta (bytes): 163262


## Bloque 4: Inspección cruda de la estructura del JSON

**Objetivo:** ver "a mano" cómo está organizada la respuesta antes de intentar convertirla en
un DataFrame.

**Justificación metodológica:** el formato de respuesta de WITS es SDMX-JSON, un estándar
distinto al de `wbgapi`. No pude ejecutar esta llamada en vivo desde mi entorno para confirmar
la forma exacta de la respuesta, así que este bloque es un paso de verificación obligatorio antes
de programar el parseo del Bloque 5 — evita que asumamos una estructura incorrecta.

**Resultado esperado:** un `print` con las claves de primer nivel del JSON (normalmente algo como
`dict_keys(['header', 'dataSets', 'structure'])` en SDMX-JSON genérico), y una muestra del contenido.

**Qué comprobar:**
- Confirmá que existan las claves `dataSets` y `structure` (son las que vamos a usar en el Bloque 5).
- Fijate dentro de `structure > dimensions` cómo se llaman las dimensiones de "series" (ahí debería
  estar el código de país/reporter) y de "observation" (ahí debería estar el año).
- Si la estructura que ves es distinta a lo que asumo en el Bloque 5, avisame con el resultado de
  este `print` y ajustamos el parseo — no sigas al Bloque 5 sin este chequeo.

In [5]:
print("Claves de primer nivel:", respuesta_json.keys())

print("\n--- structure > dimensions (así se identifican los códigos) ---")
print(json.dumps(respuesta_json.get("structure", {}).get("dimensions", {}), indent=2)[:3000])

print("\n--- Muestra de dataSets (primeras 2000 caracteres) ---")
print(json.dumps(respuesta_json.get("dataSets", {}), indent=2)[:2000])

Claves de primer nivel: dict_keys(['header', 'dataSets', 'structure'])

--- structure > dimensions (así se identifican los códigos) ---
{
  "dataset": [],
  "series": [
    {
      "id": "FREQ",
      "name": "Freq",
      "keyPosition": 0,
      "role": null,
      "values": [
        {
          "id": "A",
          "name": "Annual"
        }
      ]
    },
    {
      "id": "REPORTER",
      "name": "Reporter",
      "keyPosition": 1,
      "role": null,
      "values": [
        {
          "id": "ABW",
          "name": "Aruba"
        },
        {
          "id": "AFG",
          "name": "Afghanistan"
        },
        {
          "id": "AGO",
          "name": "Angola"
        },
        {
          "id": "AIA",
          "name": "Anguila"
        },
        {
          "id": "ALB",
          "name": "Albania"
        },
        {
          "id": "AND",
          "name": "Andorra"
        },
        {
          "id": "ANT",
          "name": "Netherlands Antilles"
        },
  

## Bloque 5: Parseo del JSON a DataFrame país-año

**Objetivo:** convertir la respuesta SDMX-JSON en un `DataFrame` con columnas `reporter_code`,
`year` y `hh_market_concentration`.


**Resultado esperado:** un DataFrame `df_market_concentration` con una fila por país-año.

In [8]:
# 1. Extraemos las dimensiones de "series" y de "observation"
dimensiones_series = respuesta_json["structure"]["dimensions"]["series"]
dimensiones_obs = respuesta_json["structure"]["dimensions"]["observation"]

# Buscamos la dimensión de país (REPORTER) y su posición real dentro de la clave
dim_reporter = next(d for d in dimensiones_series if "REPORTER" in d["id"].upper())
pos_reporter = dim_reporter["keyPosition"]  # posición real en la clave "x:x:x:x", NO el orden en la lista
codigos_reporter = [v["id"] for v in dim_reporter["values"]]

# Dimensión de tiempo (normalmente es la única dimensión de "observation")
dim_tiempo = next(d for d in dimensiones_obs if "TIME" in d["id"].upper())
codigos_anio = [v["id"] for v in dim_tiempo["values"]]

# 2. Recorremos las series usando la posición correcta para extraer el país
series = respuesta_json["dataSets"][0]["series"]

registros = []
for clave_serie, contenido in series.items():
    indices = clave_serie.split(":")
    idx_reporter = int(indices[pos_reporter])
    reporter_code = codigos_reporter[idx_reporter]

    for idx_obs, valores_obs in contenido.get("observations", {}).items():
        anio = codigos_anio[int(idx_obs)]
        valor = valores_obs[0]
        registros.append({
            "reporter_code": reporter_code,
            "year": anio,
            "hh_market_concentration": valor
        })

df_market_concentration = pd.DataFrame(registros)
df_market_concentration.head()

,reporter_code,year,hh_market_concentration
0,ABW,2000,0.768904
1,ABW,2001,0.634758
2,ABW,2002,0.575667
3,ABW,2003,0.623012
4,ABW,2004,0.540465


## Bloque 6: Verificación del resultado

**Objetivo:** revisar dimensiones, cobertura de países y años, y valores faltantes del panel
recién construido.

**Justificación metodológica:** antes de guardar el archivo, conviene confirmar que el volumen
de datos es razonable (países × años ≈ tamaño del DataFrame) y detectar de forma temprana
problemas de cobertura, siguiendo la misma lógica de verificación que usamos en el notebook 02
de disponibilidad WDI.

**Resultado esperado:** impresión de shape, cantidad de países únicos, rango de años, y
porcentaje de valores faltantes en `hh_market_concentration`.

**Qué comprobar:**
- Que `reporter_code` incluya principalmente códigos ISO3 de países (WITS también puede incluir
  agrupaciones regionales o "World" como reporters — eso se filtra más adelante en la
  estandarización, no acá, por principio de mínima intervención).
- Que el rango de años cubra razonablemente 1990-2024 (el recorte final del panel se define
  más adelante, como con V-Dem).

In [9]:
print("Dimensiones del DataFrame:", df_market_concentration.shape)
print("Países (reporter_code) únicos:", df_market_concentration["reporter_code"].nunique())
print("Rango de años:", df_market_concentration["year"].min(), "-", df_market_concentration["year"].max())

pct_faltantes = df_market_concentration["hh_market_concentration"].isna().mean() * 100
print(f"Porcentaje de valores faltantes: {pct_faltantes:.2f}%")

Dimensiones del DataFrame: (5129, 3)
Países (reporter_code) únicos: 209
Rango de años: 1988 - 2023
Porcentaje de valores faltantes: 0.00%


## Bloque 7: Exportación del panel en bruto

**Objetivo:** guardar el DataFrame tal cual, sin renombrar columnas ni filtrar países, para que
la estandarización de códigos (cruce con `df_regiones_miembros_onu`) se haga en un paso posterior
separado, igual que con WDI y V-Dem.

**Justificación metodológica:** principio de mínima intervención — este notebook solo extrae,
no transforma ni depura.

**Resultado esperado:** archivo `market_concentration_raw.xlsx` en el directorio de trabajo.

In [10]:
df_market_concentration.to_excel("market_concentration_base.xlsx", index=False)